# Solution A1 · PCA in numpy with the SVD (and with the eigenvalues)

**Type**: Implement from scratch

## Objective

Implement `fit`, `transform` and `inverse_transform` of a PCA with the SVD of the centered data, check that they match `sklearn.decomposition.PCA` (up to the sign), get the same variances with the eigenvalues of the covariance matrix and
show that the mean squared reconstruction error with `k` components is the sum of the **discarded** variances.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = StandardScaler().fit_transform(load_wine().data)
n, p = Z.shape
print(Z.shape)

## Commented solution

In [ ]:
def pca_fit(X, k):
    mean = X.mean(0)
    U, S, Vt = np.linalg.svd(X - mean, full_matrices=False)                # SVD of the centered data
    return mean, Vt[:k], S[:k] ** 2 / (len(X) - 1)                           # the rows of Vt are the components


def pca_transform(X, mean, components):
    return (X - mean) @ components.T                                         # projection onto the components


def pca_inverse(P, mean, components):
    return P @ components + mean

In [ ]:
mean, components, variances = pca_fit(Z, 3)
sk = PCA(3).fit(Z)
equal_components = bool(np.allclose(np.abs(components), np.abs(sk.components_)))
equal_variances = bool(np.allclose(variances, sk.explained_variance_))
equal_reconstruction = bool(np.allclose(pca_inverse(pca_transform(Z, mean, components), mean, components), sk.inverse_transform(sk.transform(Z))))
total_variance = Z.var(0, ddof=1).sum()
sum_of_all_variances = pca_fit(Z, 13)[2].sum()
print("equal components (up to the sign):", equal_components, "| equal variances:", equal_variances, "| equal reconstruction:", equal_reconstruction)
print(f"total variance {total_variance:.4f} = sum of the variances of the components {sum_of_all_variances:.4f}")
print("signs: does my first component have the same sign as scikit-learn's?", bool(np.sign(components[0] @ sk.components_[0]) > 0))

In [ ]:
eigenvalues = np.linalg.eigh(np.cov(Z.T))[0][::-1]                            # eigh returns the eigenvalues in increasing order
print("first three eigenvalues:", eigenvalues[:3].round(4), "| variances from the SVD:", variances.round(4))

In [ ]:
reconstruction = pca_inverse(pca_transform(Z, mean, components), mean, components)
mse = ((Z - reconstruction) ** 2).mean()
theoretical_mse = (total_variance - variances.sum()) * (n - 1) / n / p
print(f"mean squared reconstruction error with 3 components: {mse:.5f} | sum of the discarded variances (scaled): {theoretical_mse:.5f}")

## Check

In [ ]:
with verify():
    check(components.shape == (3, 13) and variances.shape == (3,), "3 components of 13 columns and 3 variances", "components: one per row")
    check(equal_components is True and equal_variances is True, "components (up to the sign) and variances match scikit-learn's PCA", "compare np.abs(components) with np.abs(sk.components_)")
    check(equal_reconstruction is True, "the reconstruction matches exactly: it does not depend on the sign", "pca_inverse(pca_transform(Z)) against sk.inverse_transform(sk.transform(Z))")
    check(close(total_variance, Z.var(0, ddof=1).sum()) and close(sum_of_all_variances, total_variance), "the sum of all the variances of the components is the total variance (13 for standardized data)", "the rotation preserves the total variance")
    check(close(eigenvalues, pca_fit(Z, 13)[2]) and (np.diff(eigenvalues) <= 0).all(), "the eigenvalues of the covariance are the variances from the SVD, in decreasing order", "np.linalg.eigh returns increasing eigenvalues: reverse them")
    check(close(mse, theoretical_mse) and close(mse, ((Z - PCA(3).fit(Z).inverse_transform(PCA(3).fit(Z).transform(Z))) ** 2).mean()), "reconstruction error = sum of the discarded variances (x (n-1)/n / p)", "total_variance - sum(variances) is the unexplained variance")
done()

## What you learned

- **PCA = centering + SVD.** The rows of `Vt` are the directions (the components), the squares of the singular values divided by `n - 1` are the variances: the same thing you get from the **eigenvectors and eigenvalues of the covariance matrix**. With the SVD it is numerically
  more stable (it never forms the covariance matrix).
- **The sign is arbitrary:** `v` and `-v` describe the same direction. The components of this implementation and of scikit-learn coincide up to the sign; the reconstruction, which does not depend on it, coincides exactly. Never compare signs.
- The rotation **preserves the total variance** (13 for standardized data) and distributes it in decreasing order among the components. That is why "explained variance" and "reconstruction error" are the same information: the mean squared error with `k` components is the variance you discarded.
- With `k` components the reconstruction is the **best rank-`k` approximation** (Eckart-Young theorem): no other projection onto `k` dimensions does better in squared error.

## Common pitfall

Forgetting to center (or to add back the mean in `inverse_transform`): the SVD of uncentered data puts in the first component the direction of the mean, not that of maximum variance (see E2).

## Interview question

> *Click on a question to show its answer.*

<details><summary>How is PCA computed and what is the relationship with the eigenvalues?</summary>

You center the data and compute the SVD of the centered matrix X = U S V^T: the rows of V^T are the components and S^2/(n-1) are the variances. It is equivalent to the spectral decomposition of the covariance (the eigenvectors are the components, the eigenvalues the variances), but the SVD is
numerically more stable and for large matrices truncated or randomized variants are used. The projection onto the first k components is the best rank-k approximation in squared error.

</details>